<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_5d_b2_mu7_4_mmnn_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Five-player Cournot game with MMNN: stochastic DTB and Euler--Maruyama

This notebook advances the MMNN parameters and particle values at every step. The evolving MMNN supplies the tangent field; particle values are accumulated separately and are never replaced by a neural-map evaluation. There are no resets or NN refits.

With $b=2$, $\mu=7/4$, and $s_i=\sum_{j\ne i}x_j$, the drift is

$$F_i(x)=2b\left(\max\{\mu s_i(1-s_i),0\}-x_i\right).$$

The stochastic game is

$$dX_t=F(X_t)\,dt+\Sigma\,dW_t,\qquad D=\Sigma\Sigma^\top.$$

DTB evolves the same probability law with the deterministic probability-flow target

$$g(x,t)=F(x)-\frac12Dq(x,t),\qquad q=\nabla_x\log\rho.$$

The DTB particles therefore do **not** receive sampled Brownian increments: the score correction already represents diffusion in the density equation. The independent Euler--Maruyama reference receives $\Sigma\sqrt h\,\xi_k$. Adding both mechanisms to DTB would count the same diffusion twice.

The default initial law is a lightly smoothed uniform cloud, $U([0,1]^5)+N(0,\tau^2I)$, because it has an analytical finite score. Set `INITIAL_LAW = 'uniform'` to recover exact uniform samples and the interior-score approximation $q_0=0$; the exact uniform boundary score is singular.

MMNN duplicate of the [original deterministic notebook](https://github.com/sun-mengwei/dtb-colab-experiments/blob/codex/game-dynamics-dtb/DTB_Game_Ver2/cournot_5d_b2_mu7_4_dtb.ipynb). Run all cells for matched DTB and Euler--Maruyama clouds in the four adjacent coordinate planes.

In [ ]:
from pathlib import Path
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Use the existing helpers locally, or obtain them when running in Colab.
module_files = ('dtb.py', 'run_game_dtb.py', 'network.py', 'utility.py')
module_dir = next((p for p in (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2')
                   if all((p / name).is_file() for name in module_files)), None)
if module_dir is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run from the repository root or DTB_Game_Ver2.')
    repo = Path('/content/dtb-colab-experiments')
    if not repo.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo),
        ], check=True)
    module_dir = repo / 'DTB_Game_Ver2'
sys.path.insert(0, str(module_dir.resolve()))

from dtb import flat_params
from run_game_dtb import game_dtb_basis_matrices
from network import MMNN
from utility import (
    euler_score_update,
    sample_initial_with_score,
    tangent_velocity_spatial_terms,
)

output_root = Path('/content') if Path('/content').is_dir() else module_dir
output_dir = output_root / 'cournot_5d_mmnn_results'
output_dir.mkdir(parents=True, exist_ok=True)

## Parameters

The Cournot drift, sample count, time grid, MMNN, 64-coordinate tangent solve, and SVD cutoff match the deterministic MMNN notebook. Five independent noise components have standard deviation `0.1`, so $D=0.01I_5$.

`INITIAL_LAW='smoothed_uniform'` supplies an analytical boundary-aware score while remaining close to the original uniform cloud. Both DTB and Euler--Maruyama start from the exact same sampled cloud. `DERIVATIVE_CHUNK` only controls memory use when evaluating $D_xu$ and $\nabla_x\operatorname{div}u$.

In [ ]:
SEED = 0
EM_SEED = SEED + 2
DIM = 5
COURNOT_B = 2.0
COURNOT_MU = 7 / 4
N_PARTICLES = 3000

NOISE_STD = (0.1,) * DIM
INITIAL_LAW = 'smoothed_uniform'  # Use 'uniform' for exact uniform samples and q_0=0 in the interior.
SMOOTHING_STD = 0.02

H = 0.005
T_FINAL = 2
SNAPSHOT_TIMES = (0.0, 0.5, 1.0, T_FINAL)
WIDTH, RANK, DEPTH = 16, 16, 2
BASIS_SIZE = 64
SVD_RTOL = 1e-6  # Retain singular values above SVD_RTOL * s_max.
JACOBIAN_CHUNK = 256
DERIVATIVE_CHUNK = 32

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
torch.set_num_threads(min(4, torch.get_num_threads()))

if INITIAL_LAW not in ('uniform', 'smoothed_uniform'):
    raise ValueError("INITIAL_LAW must be 'uniform' or 'smoothed_uniform'.")
if len(NOISE_STD) != DIM or any(not math.isfinite(s) or s < 0 for s in NOISE_STD):
    raise ValueError('NOISE_STD must contain one finite nonnegative value per coordinate.')
if SMOOTHING_STD <= 0 or min(H, T_FINAL) <= 0:
    raise ValueError('SMOOTHING_STD, H, and T_FINAL must be positive.')


def game_velocity(x):
    rivals = x.sum(dim=-1, keepdim=True) - x
    best_response = (COURNOT_MU * rivals * (1 - rivals)).clamp_min(0.0)
    return 2 * COURNOT_B * (best_response - x)

## Evolve parameters, particle values, and attached scores

At step $k$, choose the same kind of 64-coordinate parameter subset as in the deterministic notebook and evaluate the neural tangent basis at the current physical particles:

$$J_i^k=\partial_{\theta_{S_k}}T_{\theta_k}(x_i^k),\qquad
g_i^k=F(x_i^k)-\frac12Dq_i^k.$$

The truncated-SVD solve projects $g^k$ onto $J^k$. With $u_i^k=J_i^k\alpha^k$, update

$$\theta_{k+1}=\theta_k+h_kE_{S_k}\alpha^k,
\qquad x_i^{k+1}=x_i^k+h_ku_i^k.$$

The new particle values are not obtained from $T_{\theta_{k+1}}$. The NN parameters only produce the next tangent field.

For the attached score and log density, differentiate the same old-step tangent field in physical coordinates and use

$$q_i^{k+1}=q_i^k-h_k\left[(D_xu_i^k)^\top q_i^k+
\nabla_x\operatorname{div}u_i^k\right],\qquad
\ell_i^{k+1}=\ell_i^k-h_k\operatorname{div}u_i^k.$$

`X_final` and `Q_final` store the final particles and scores. The projection error compares $J^k\alpha^k$ with the complete score-corrected target $g^k$.

In [ ]:
class ResidualMMNNMap(torch.nn.Module):
    """T_theta(x) = x + MMNN_theta(x), initialized to the identity."""

    def __init__(self, dim, width, rank, depth, activation='tanh',
                 dtype=torch.float64):
        super().__init__()
        self.net = MMNN(d_in=dim, width=width, rank=rank, depth=depth,
                        d_out=dim, activation=activation, dtype=dtype)
        # W,b remain frozen; only A,c participate in the DTB parameter vector.
        with torch.no_grad():
            self.net.layers[-1].A.zero_()
            self.net.layers[-1].c.zero_()

    def forward(self, x):
        return x + self.net(x)


# Rerunning this cell restarts DTB and leaves the later EM reference independent.
torch.manual_seed(SEED)
initial_generator = torch.Generator().manual_seed(SEED)
z_cpu, q_cpu, log_density_cpu = sample_initial_with_score(
    N_PARTICLES, DIM, law=INITIAL_LAW, device=torch.device('cpu'), dtype=DTYPE,
    generator=initial_generator, smoothing_std=SMOOTHING_STD,
)
z = z_cpu.to(DEVICE)
x = z.clone()
score = q_cpu.to(DEVICE)
log_density = log_density_cpu.to(DEVICE)

sigma = torch.tensor(NOISE_STD, device=DEVICE, dtype=DTYPE)
diffusion = torch.diag(sigma.square())
model = ResidualMMNNMap(dim=DIM, width=WIDTH, rank=RANK, depth=DEPTH,
                        activation='tanh', dtype=DTYPE).to(DEVICE)
theta, structure, _ = flat_params(model)
basis_generator = torch.Generator().manual_seed(SEED + 1)

snapshot_times = np.unique(np.round(SNAPSHOT_TIMES, 12))
times = np.unique(np.round(np.r_[np.arange(0, T_FINAL, H), snapshot_times, T_FINAL], 12))
projection_times = times[:-1]

projection_error = []
score_rms = []
diffusion_rms = []
jacobian_sigma_max = []
jacobian_sigma_min = []
jacobian_condition = []
clouds = {0.0: x.cpu().numpy().copy()}

for k, t in enumerate(projection_times):
    drift = game_velocity(x)
    diffusion_correction = 0.5 * (score @ diffusion.T)
    target_velocity = drift - diffusion_correction
    score_rms.append(score.square().sum(dim=1).mean().sqrt().item())
    diffusion_rms.append(
        diffusion_correction.square().sum(dim=1).mean().sqrt().item()
    )

    indices = torch.randperm(theta.numel(), generator=basis_generator)
    selected = indices[:BASIS_SIZE].sort().values.to(DEVICE)
    theta_step = theta.detach().clone()
    _, _, J_flat = game_dtb_basis_matrices(
        theta_step, selected, x, model, structure, chunk=JACOBIAN_CHUNK)
    J_flat = J_flat.detach()

    A = J_flat / np.sqrt(N_PARTICLES)
    target = target_velocity.reshape(-1) / np.sqrt(N_PARTICLES)
    U, s, Vh = torch.linalg.svd(A, full_matrices=False)
    sigma_max, sigma_min = s[0].item(), s[-1].item()
    jacobian_sigma_max.append(sigma_max * np.sqrt(N_PARTICLES))
    jacobian_sigma_min.append(sigma_min * np.sqrt(N_PARTICLES))
    jacobian_condition.append(sigma_max / sigma_min if sigma_min > 0 else np.inf)
    keep = s > SVD_RTOL * s[0]
    if not keep.any():
        raise FloatingPointError(f'No tangent singular value retained at t={t:g}.')
    alpha = Vh[keep].T @ ((U[:, keep].T @ target) / s[keep])

    tangent_velocity, grad_u, divergence, grad_divergence = (
        tangent_velocity_spatial_terms(
            theta_step, selected, alpha, x, model, structure,
            chunk_size=DERIVATIVE_CHUNK,
        )
    )
    error = (tangent_velocity - target_velocity).square().sum(dim=1).mean().sqrt()
    if not torch.isfinite(error):
        raise FloatingPointError(f'Nonfinite tangent projection at t={t:g}.')
    projection_error.append(error.item())

    next_time = float(times[k + 1])
    h = next_time - float(t)
    next_score, _, _ = euler_score_update(score, grad_u, grad_divergence, h)
    x = (x + h * tangent_velocity).detach()
    score = next_score.detach()
    log_density = (log_density - h * divergence).detach()
    theta = theta_step.clone()
    theta[selected] += h * alpha
    if not all(torch.isfinite(value).all() for value in
               (theta, x, score, log_density)):
        raise FloatingPointError(f'Nonfinite state at t={next_time:g}.')

    if next_time in snapshot_times:
        clouds[next_time] = x.cpu().numpy().copy()

X_final = x.detach().clone()
Q_final = score.detach().clone()
Log_density_final = log_density.detach().clone()
score_rms.append(Q_final.square().sum(dim=1).mean().sqrt().item())
projection_error = np.asarray(projection_error)
score_rms = np.asarray(score_rms)
diffusion_rms = np.asarray(diffusion_rms)
jacobian_sigma_max = np.asarray(jacobian_sigma_max)
jacobian_sigma_min = np.asarray(jacobian_sigma_min)
jacobian_condition = np.asarray(jacobian_condition)

np.save(output_dir / 'dtb_final_particles.npy', X_final.cpu().numpy())
np.save(output_dir / 'dtb_final_scores.npy', Q_final.cpu().numpy())
np.save(output_dir / 'dtb_final_log_density.npy', Log_density_final.cpu().numpy())
np.savetxt(
    output_dir / 'stochastic_dtb_diagnostics.csv',
    np.column_stack((projection_times, projection_error, score_rms[:-1], diffusion_rms)),
    delimiter=',', header='time,projection_error,score_rms,diffusion_velocity_rms', comments='',
)
print(f'DTB complete: sigma={tuple(float(v) for v in sigma)}, '
      f'final score RMS={score_rms[-1]:.6g}.')
if INITIAL_LAW == 'uniform':
    print('Uniform-score note: q_0=0 is only the interior score; the boundary score is singular.')

## DTB point-cloud projections

Rows show the four adjacent coordinate pairs; columns show physical time. These are the independently accumulated stochastic probability-flow particles. All particles are displayed with common axis limits.

In [ ]:
pairs = ((0, 1), (1, 2), (2, 3), (3, 4))
fig, axes = plt.subplots(4, len(clouds), figsize=(3.5 * len(clouds), 11),
                         sharex=True, sharey=True, squeeze=False, layout='constrained')
lo = min(points.min() for points in clouds.values())
hi = max(points.max() for points in clouds.values())
padding = 0.04 * max(hi - lo, 1e-6)

for row, (i, j) in enumerate(pairs):
    for col, (t, points) in enumerate(clouds.items()):
        ax = axes[row, col]
        ax.scatter(points[:, i], points[:, j], s=3, alpha=0.35,
                   color='#176b87', linewidths=0, rasterized=True)
        ax.set(xlabel=fr'$x_{i + 1}$', ylabel=fr'$x_{j + 1}$',
               xlim=(lo - padding, hi + padding), ylim=(lo - padding, hi + padding))
        ax.set_aspect('equal', adjustable='box')
        if row == 0:
            ax.set_title(f't = {t:g}')
fig.suptitle(
    fr'Five-player stochastic DTB: $b={COURNOT_B:g}$, $\mu={COURNOT_MU:g}$, '
    fr'$\sigma_i={NOISE_STD[0]:g}$'
)
fig.savefig(output_dir / 'point_clouds.png', dpi=300, bbox_inches='tight')
plt.show()

## Projection-error trajectory

The residual is measured against the complete target $F(X_k)-\tfrac12Dq_k$.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
ax.plot(projection_times, projection_error, color='#176b87', linewidth=1.5)
ax.set(xlabel='Time', ylabel='RMS projection residual',
       title='Score-corrected tangent-projection error', xlim=(0, T_FINAL), ylim=(0, None))
ax.grid(alpha=0.2)
fig.savefig(output_dir / 'projection_error.png', dpi=300, bbox_inches='tight')
plt.show()

## Jacobian condition-number trajectory

At the start of each DTB step, record

$$\kappa_2(J^k)=\frac{\sigma_{\max}(J^k)}{\sigma_{\min}(J^k)}.$$

`J_flat` has shape `(N_PARTICLES * DIM, BASIS_SIZE)` for that step's selected parameter coordinates. The same SVD used by the score-corrected projection supplies this diagnostic before `SVD_RTOL` filtering. Infinite values mean the computed minimum singular value is exactly zero.

In [ ]:
# Run after the DTB evolution cell to use its recorded singular values.
condition_finite = np.isfinite(jacobian_condition)
condition_infinite = np.isinf(jacobian_condition)
condition_fig, condition_ax = plt.subplots(figsize=(8, 4), layout='constrained')
condition_ax.plot(
    projection_times,
    np.where(condition_finite, jacobian_condition, np.nan),
    color='#7c3aed', linewidth=1.2,
    label=r'$\sigma_{\max}/\sigma_{\min}$',
)
if not condition_finite.any():
    condition_ax.set_ylim(1, 10)
condition_ax.set_yscale('log')
if condition_infinite.any():
    condition_ax.scatter(
        projection_times[condition_infinite],
        np.full(condition_infinite.sum(), 0.96),
        transform=condition_ax.get_xaxis_transform(),
        color='#dc2626', marker='x', s=20,
        label=r'$\infty$ (zero minimum; marked at top)', zorder=3,
    )
condition_ax.set(
    xlabel='Time', ylabel=r'Condition number $\kappa_2(J)$ (log scale)',
    title='Selected Jacobian condition number', xlim=(0, T_FINAL),
)
condition_ax.grid(alpha=0.2, which='both')
condition_ax.legend(loc='best', fontsize=9)
condition_fig.savefig(
    output_dir / 'jacobian_condition_number.png', dpi=300, bbox_inches='tight',
)
np.savetxt(
    output_dir / 'jacobian_condition_number.csv',
    np.column_stack((projection_times, jacobian_sigma_max,
                     jacobian_sigma_min, jacobian_condition)),
    delimiter=',', header='time,sigma_max,sigma_min,condition_number', comments='',
)
print(f'Recorded {len(jacobian_condition)} Jacobian condition numbers; '
      f'{condition_infinite.sum()} infinite.')
plt.show()


## Independent Euler--Maruyama reference on the same initial cloud and time grid

Euler--Maruyama uses the same $z_i$, every step size, and every snapshot time as DTB:

$$X_{k+1}^{\mathrm{EM}}=X_k^{\mathrm{EM}}+h_kF(X_k^{\mathrm{EM}})+\Sigma\sqrt{h_k}\,\xi_k,
\qquad \xi_k\sim N(0,I_5).$$

Its private seeded generator makes reruns reproducible without changing DTB sampling or tangent selection. EM Brownian paths and DTB probability-flow paths do not correspond particle by particle, so the printed paired-label RMS is descriptive; the point clouds are the distributional comparison.

In [ ]:
# Run after the DTB evolution cell; rerunning starts from the same z and noise seed.
x_em = z.detach().clone()
em_generator = torch.Generator().manual_seed(EM_SEED)
clouds_em = {float(times[0]): x_em.cpu().numpy().copy()}
em_snapshot_times = set(float(value) for value in snapshot_times)

with torch.no_grad():
    for em_t, em_next_time in zip(times[:-1], times[1:]):
        em_h = float(em_next_time - em_t)
        noise = torch.randn(x_em.shape, dtype=DTYPE, generator=em_generator).to(DEVICE)
        x_em = (
            x_em + em_h * game_velocity(x_em)
            + math.sqrt(em_h) * sigma * noise
        )
        if not torch.isfinite(x_em).all():
            raise FloatingPointError(f'Nonfinite EM state at t={em_next_time:g}.')
        if float(em_next_time) in em_snapshot_times:
            clouds_em[float(em_next_time)] = x_em.cpu().numpy().copy()

X_em_final = x_em.detach().clone()
em_final_rms_distance = (
    (X_final - X_em_final).square().sum(dim=1).mean().sqrt().item()
)
em_final_mean_distance = torch.linalg.vector_norm(
    X_final.mean(dim=0) - X_em_final.mean(dim=0)
).item()
np.save(output_dir / 'em_final_particles.npy', X_em_final.cpu().numpy())
print(f'EM reference: {z.shape[0]} particles, {len(times) - 1} steps, '
      f'nominal H={H:g}, final time={times[-1]:g}, sigma={tuple(float(v) for v in sigma)}.')
print(f'Final sample-mean distance (DTB vs EM): {em_final_mean_distance:.6g}')
print(f'Final paired-label RMS (descriptive only): {em_final_rms_distance:.6g}')

## Euler--Maruyama reference: four projection planes

Rows show $(x_1,x_2)$, $(x_2,x_3)$, $(x_3,x_4)$, and $(x_4,x_5)$; columns show the same snapshot times as the DTB figure. Common axis limits cover both methods.

In [ ]:
em_pairs = ((0, 1), (1, 2), (2, 3), (3, 4))
em_fig, em_axes = plt.subplots(
    4, len(clouds_em), figsize=(3.5 * len(clouds_em), 11),
    sharex=True, sharey=True, squeeze=False, layout='constrained',
)
em_all_clouds = list(clouds.values()) + list(clouds_em.values())
em_lo = min(points.min() for points in em_all_clouds)
em_hi = max(points.max() for points in em_all_clouds)
em_padding = 0.04 * max(em_hi - em_lo, 1e-6)

for em_row, (em_i, em_j) in enumerate(em_pairs):
    for em_col, (em_time, em_points) in enumerate(clouds_em.items()):
        em_ax = em_axes[em_row, em_col]
        em_ax.scatter(em_points[:, em_i], em_points[:, em_j], s=3, alpha=0.35,
                      color='#b45309', linewidths=0, rasterized=True)
        em_ax.set(xlabel=fr'$x_{em_i + 1}$', ylabel=fr'$x_{em_j + 1}$',
                  xlim=(em_lo - em_padding, em_hi + em_padding),
                  ylim=(em_lo - em_padding, em_hi + em_padding))
        em_ax.set_aspect('equal', adjustable='box')
        if em_row == 0:
            em_ax.set_title(f't = {em_time:g}')
em_fig.suptitle(
    fr'Five-player Euler--Maruyama: $b={COURNOT_B:g}$, '
    fr'$\mu={COURNOT_MU:g}$, $\sigma_i={NOISE_STD[0]:g}$'
)
em_fig.savefig(output_dir / 'em_point_clouds.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from pathlib import Path
import shutil

run_folder = Path(output_dir)
zip_path = shutil.make_archive(
    str(output_root / run_folder.name),
    'zip',
    root_dir=str(run_folder.parent),
    base_dir=run_folder.name,
)

try:
    from google.colab import files
except ImportError:
    print('Saved result archive:', zip_path)
else:
    files.download(zip_path)